# Unsloth Qwen3 LoRA/QLoRA SFT + GRPO 实验模板

这个 notebook 按 Unsloth 官方 conversational notebook 的写法组织：

1. `FastLanguageModel.from_pretrained` 加载 4-bit 模型。
2. `FastLanguageModel.get_peft_model` 注入 LoRA。
3. 用 TRL `SFTTrainer` 做 SFT。
4. 用 Unsloth + TRL `GRPOTrainer` 做一个最小 RL/GRPO 示例。
5. 保存 LoRA adapter，并演示单条与批量推理。

> 如果你只做 SFT，可以只运行到“保存 SFT adapter”。如果你要接着做 RL，再运行 GRPO 部分。


## 0. 安装依赖

服务器上建议先按仓库根目录执行：

```bash
pip install -r requirements.txt
```

如果在 Colab 或临时 notebook 环境里，可以取消下面 cell 的注释安装。


In [ ]:
# %%capture
# !pip install -r ../requirements.txt


## 1. 基础配置

默认使用 `unsloth/Qwen3-1.7B-bnb-4bit`。如果你已经下载到本地，可以把 `model_name` 改成 `../models/Qwen3-1.7B-bnb-4bit`。


In [ ]:
from pathlib import Path

max_seq_length = 1024
dtype = None          # None = Unsloth 自动选择；也可设 torch.float16 / torch.bfloat16
load_in_4bit = True

model_name = "unsloth/Qwen3-1.7B-bnb-4bit"
train_file = Path("../data/toy_sft.jsonl")
output_dir = Path("../outputs/qwen3_1p7b_unsloth_notebook_lora")


## 2. 加载模型并注入 LoRA

这部分基本就是官方 Unsloth notebook 的核心写法：先 `from_pretrained`，再 `get_peft_model`。


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 3. 准备 SFT 数据

仓库支持：

- `{"prompt": "...", "groundtruth": "..."}`
- `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}`
- `{"text": "..."}`

为了贴近官方 Unsloth notebook，这里先把样本渲染成 `text` 列并用 TRL `SFTTrainer` 训练。若你想做 response-only loss，请直接用仓库脚本 `scripts/train_lora.py`（默认会 mask prompt）。


In [ ]:
import sys
sys.path.insert(0, str((Path("..") / "src").resolve()))

from llm_lab.data import load_sft_dataset
train_dataset = load_sft_dataset(
    train_file,
    tokenizer,
    prompt_field = "prompt",
    response_field = "groundtruth",
    response_only_loss = False,
)
train_dataset


## 4. SFT 训练

这里按官方 Unsloth notebook 风格使用 TRL `SFTTrainer`。


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir = str(output_dir),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    save_strategy = "epoch",
    report_to = "none",
    dataset_text_field = "text",
    max_length = max_seq_length,
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    args = training_args,
    train_dataset = train_dataset,
)

trainer.train()


## 5. 保存 SFT adapter

默认保存 LoRA adapter；如果需要部署合并模型，可以使用 `save_pretrained_merged`。


In [ ]:
model.save_pretrained(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
print(f"Saved LoRA adapter to {output_dir}")

# 可选：导出合并后的 16-bit 模型
# model.save_pretrained_merged(str(output_dir) + "_merged_16bit", tokenizer, save_method = "merged_16bit")


## 6. 单条推理


In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "请用一句话解释什么是大语言模型。"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 0.7,
    top_p = 0.9,
    use_cache = True,
)
print(tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[-1]:], skip_special_tokens=True)[0])


## 7. 批量推理

Notebook 里给一个简单版本；正式批量跑建议用仓库脚本：

```bash
CUDA_VISIBLE_DEVICES=0 python ../scripts/batch_infer_lora.py \
  --model_name_or_path ../outputs/qwen3_1p7b_unsloth_notebook_lora \
  --input_file ../data/prompts.json \
  --output_file ../outputs/notebook_batch_outputs.json \
  --batch_size 4 \
  --num_repeats 1 \
  --overwrite
```


In [ ]:
import json

rows = json.loads(Path("../data/prompts.json").read_text(encoding="utf-8"))
texts = []
for row in rows[:2]:
    messages = [{"role": "user", "content": row["prompt"]}]
    texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

inputs = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9, use_cache=True)
responses = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[-1]:], skip_special_tokens=True)
responses


## 8. 可选：GRPO / RL 训练最小模板

下面是最小 GRPO 示例。真实实验时，建议把 `reward_func` 换成你的任务 reward。


In [ ]:
# 如果只做 SFT，可以跳过这个 cell。
from unsloth import PatchFastRL
PatchFastRL("grpo", FastLanguageModel)

from trl import GRPOConfig, GRPOTrainer
from llm_lab.data import load_grpo_dataset

rl_dataset = load_grpo_dataset(
    train_file,
    prompt_field = "prompt",
    answer_field = "groundtruth",
)

def reward_func(completions, answer, **kwargs):
    rewards = []
    for completion, expected in zip(completions, answer):
        if isinstance(completion, list):
            generated = "\n".join(item.get("content", str(item)) if isinstance(item, dict) else str(item) for item in completion)
        else:
            generated = str(completion)
        rewards.append(1.0 if expected.strip().lower() in generated.strip().lower() else 0.0)
    return rewards

grp_args = GRPOConfig(
    output_dir = str(output_dir) + "_grpo",
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 2,
    max_prompt_length = 768,
    max_completion_length = 256,
    max_steps = 20,
    beta = 0.0,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    report_to = "none",
)

grp_trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [reward_func],
    args = grp_args,
    train_dataset = rl_dataset,
)

# grp_trainer.train()
